In [1]:
from cobra.io import read_sbml_model

model = read_sbml_model("../models/iJR904.xml.gz")

print(model)
print("Reactions:", len(model.reactions))
print("Metabolites:", len(model.metabolites))
print("Genes:", len(model.genes))

iJR904
Reactions: 1075
Metabolites: 761
Genes: 904


In [2]:
print("Objective:")
print(model.objective)

print("\nSolver:")
print(model.solver.interface)

print("\nCurrent medium:")
model.medium

Objective:
Maximize
1.0*BIOMASS_Ecoli - 1.0*BIOMASS_Ecoli_reverse_bf7a1

Solver:
<module 'optlang.glpk_interface' from '/home/jupyter-852758/.conda/envs/cobra_env/lib/python3.11/site-packages/optlang/glpk_interface.py'>

Current medium:


{'EX_h2o_e': 999999.0,
 'EX_h_e': 999999.0,
 'EX_co2_e': 999999.0,
 'EX_k_e': 999999.0,
 'EX_fe2_e': 999999.0,
 'EX_glc__D_e': 10.0,
 'EX_na1_e': 999999.0,
 'EX_nh4_e': 999999.0,
 'EX_o2_e': 20.0,
 'EX_pi_e': 999999.0,
 'EX_so4_e': 999999.0}

In [3]:
for rxn in model.exchanges:
    text = (rxn.id + " " + rxn.name).lower()
    if any(x in text for x in ["glucose", "maltose", "galactose",
                               "glycerol", "lactate", "acetate"]):
        print(rxn.id, "|", rxn.name)

EX_ac_e | Acetate exchange
EX_acac_e | Acetoacetate exchange
EX_lac__D_e | D-lactate exchange
EX_lac__L_e | L-Lactate exchange
EX_g6p_e | D-Glucose 6-phosphate exchange
EX_gal_e | D-Galactose exchange
EX_malt_e | Maltose exchange
EX_glc__D_e | D-Glucose exchange
EX_glyc3p_e | Glycerol 3-phosphate exchange
EX_glyc_e | Glycerol exchange


In [4]:
base_medium = model.medium.copy()

carbon_sources = [
    "EX_glc__D_e",
    "EX_malt_e",
    "EX_gal_e",
    "EX_glyc_e",
    "EX_lac__L_e",
    "EX_ac_e"
]

# alle Carbon Sources zunächst entfernen
for rxn_id in carbon_sources:
    base_medium.pop(rxn_id, None)

base_medium

{'EX_h2o_e': 999999.0,
 'EX_h_e': 999999.0,
 'EX_co2_e': 999999.0,
 'EX_k_e': 999999.0,
 'EX_fe2_e': 999999.0,
 'EX_na1_e': 999999.0,
 'EX_nh4_e': 999999.0,
 'EX_o2_e': 20.0,
 'EX_pi_e': 999999.0,
 'EX_so4_e': 999999.0}

In [5]:
carbon_conditions = {
    "Glucose":   ("EX_glc__D_e", 10.0),
    "Maltose":   ("EX_malt_e", 5.0),
    "Galactose": ("EX_gal_e", 10.0),
    "Glycerol":  ("EX_glyc_e", 20.0),
    "Lactate":   ("EX_lac__L_e", 20.0),
    "Acetate":   ("EX_ac_e", 30.0),
}

In [6]:
results = {}

for carbon, (exchange_id, uptake) in carbon_conditions.items():

    medium = base_medium.copy()
    medium[exchange_id] = uptake

    with model:
        model.medium = medium
        solution = model.optimize()

        results[carbon] = solution.objective_value

for carbon, growth in results.items():
    print(f"{carbon:10s}: {growth:.3f} h^-1")

Glucose   : 0.922 h^-1
Maltose   : 0.922 h^-1
Galactose : 0.899 h^-1
Glycerol  : 0.988 h^-1
Lactate   : 0.495 h^-1
Acetate   : 0.407 h^-1


In [7]:
base_medium = model.medium.copy()

base_medium.pop("EX_glc__D_e", None)

print(base_medium)

{'EX_h2o_e': 999999.0, 'EX_h_e': 999999.0, 'EX_co2_e': 999999.0, 'EX_k_e': 999999.0, 'EX_fe2_e': 999999.0, 'EX_na1_e': 999999.0, 'EX_nh4_e': 999999.0, 'EX_o2_e': 20.0, 'EX_pi_e': 999999.0, 'EX_so4_e': 999999.0}


In [8]:
import pandas as pd

df_fba = pd.DataFrame(results).T
df_fba

ValueError: If using all scalar values, you must pass an index